<a href="https://colab.research.google.com/github/nejumi/weave-initial-course/blob/main/notebooks/02_practice/01_simple_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04. シンプルエージェント：OpenAI Function Calling + Weave

OpenAI の Function Calling を使ったツール利用エージェントを構築し、
`@weave.op` で各ツール呼び出しをトレースします。

## 学習内容
- OpenAI Function Calling の基本
- ツール関数を `@weave.op` でトレース
- エージェントループ全体のトレース階層
- Weave UI でのデバッグ方法


In [ ]:
# ローカル環境: uv sync --all-extras で一括インストール可能
# Colab: 以下のセルで個別インストール
!pip install -q uv
!uv pip install --system -q \
  "weave>=0.51.0" \
  "wandb>=0.19.0" \
  "openai>=1.0.0" \
  "python-dotenv>=1.0.0"

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    # WANDB_BASE_URL: 値がある場合のみセット（空 / 未設定 → SaaS デフォルト）
    _base_url = (userdata.get("WANDB_BASE_URL") or "").strip()
    if _base_url:
        os.environ["WANDB_BASE_URL"] = _base_url
    os.environ.setdefault("WANDB_API_KEY",  userdata.get("WANDB_API_KEY"))
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
    os.environ.setdefault("WANDB_ENTITY",   userdata.get("WANDB_ENTITY"))
    os.environ.setdefault("WANDB_PROJECT",  userdata.get("WANDB_PROJECT") or "weave-course")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv; load_dotenv()
    # ローカル: .env に空欄で書かれていた場合も削除
    if not os.environ.get("WANDB_BASE_URL", "").strip():
        os.environ.pop("WANDB_BASE_URL", None)

ENTITY  = os.environ["WANDB_ENTITY"]
PROJECT = os.environ.get("WANDB_PROJECT", "weave-course")
_target = os.environ.get("WANDB_BASE_URL", "https://api.wandb.ai (SaaS)")
print(f"Entity : {ENTITY}")
print(f"Project: {PROJECT}")
print(f"W&B URL: {_target}")


In [ ]:
import json
import weave
from openai import OpenAI

client_weave = weave.init(f"{ENTITY}/{PROJECT}")
oai = OpenAI()


## 1. ツール定義

3 つのツールを用意します。それぞれ `@weave.op()` でトレース対象にします。


In [ ]:
import math

# ── ツール実装（@weave.op でトレース） ──────────────────────────────────────

@weave.op()
def calculate(expression: str) -> dict:
    """数式を評価して結果を返す。例: '2 ** 10', 'math.sqrt(144)'"""
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return {"result": float(result), "error": None}
    except Exception as e:
        return {"result": None, "error": str(e)}

@weave.op()
def get_weather(city: str) -> dict:
    """指定都市の天気を返す（デモ用モック）"""
    mock_data = {
        "tokyo":    {"temp_c": 22, "condition": "Sunny",  "humidity": 55},
        "osaka":    {"temp_c": 24, "condition": "Cloudy", "humidity": 60},
        "sapporo":  {"temp_c": 12, "condition": "Rainy",  "humidity": 80},
        "fukuoka":  {"temp_c": 26, "condition": "Sunny",  "humidity": 65},
    }
    key = city.lower().replace(" ", "")
    return mock_data.get(key, {"temp_c": 20, "condition": "Unknown", "humidity": 50})

@weave.op()
def search_web(query: str) -> dict:
    """ウェブ検索を実行する（デモ用モック）"""
    mock_results = {
        "w&b weave": "W&B Weave is an LLM observability platform by Weights & Biases.",
        "openai function calling": "OpenAI Function Calling allows models to call developer-defined functions.",
        "grpo": "GRPO (Group Relative Policy Optimization) is an RL algorithm for LLM training.",
    }
    for key, val in mock_results.items():
        if key in query.lower():
            return {"snippet": val, "source": "mock_search"}
    return {"snippet": f"No results found for: {query}", "source": "mock_search"}

# ── OpenAI ツール定義スキーマ ────────────────────────────────────────────────
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Use Python syntax, e.g. '2**10', 'math.sqrt(16)'",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "Python math expression"}},
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a Japanese city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name in English, e.g. tokyo"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the web for information",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Search query"}},
                "required": ["query"],
            },
        },
    },
]

TOOL_MAP = {"calculate": calculate, "get_weather": get_weather, "search_web": search_web}
print("ツール定義完了")


## 2. エージェントループ

エージェント本体も `@weave.op()` で囲むことで、
ツール呼び出しが子スパンとして階層表示されます。


In [ ]:
@weave.op()
def run_agent(user_message: str, max_turns: int = 6) -> str:
    """Function Calling エージェントのメインループ"""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant with access to tools. "
            "Use tools when needed to answer accurately. "
            "Answer in Japanese."
        )},
        {"role": "user", "content": user_message},
    ]

    for turn in range(max_turns):
        resp = oai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        msg = resp.choices[0].message
        messages.append(msg)

        # ツール呼び出しがなければ終了
        if not msg.tool_calls:
            return msg.content

        # ツールを実行して結果を追加
        for tc in msg.tool_calls:
            func_name = tc.function.name
            args = json.loads(tc.function.arguments)
            tool_result = TOOL_MAP[func_name](**args)
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(tool_result, ensure_ascii=False),
            })

    return messages[-1].get("content", "最大ターン数に達しました")

# 動作確認
print(run_agent("東京の気温の2倍は何度ですか？"))


## 3. 複数クエリで実行してトレースを蓄積

In [ ]:
questions = [
    "2の10乗はいくつですか？",
    "東京と大阪の天気を教えてください",
    "144の平方根は何ですか？",
    "W&B Weaveとは何ですか？",
    "東京の気温の摂氏をケルビンに変換してください（ケルビン = 摂氏 + 273.15）",
]

for q in questions:
    print()
    print(f"質問: {q}")
    answer = run_agent(q)
    print(f"回答: {answer}")


## 4. Weave UI でトレースを確認

上記の実行後、Weave UI で以下を確認してください：

1. **Traces タブ** → `run_agent` が root、`calculate` / `get_weather` / `search_web` が子スパン
2. **各 call の詳細** → 入力・出力・レイテンシ・トークン数
3. **ネスト構造** → 何ターンで応答したか


In [ ]:
print(f"Weave UI: https://wandb.ai/{ENTITY}/{PROJECT}/weave/traces")


## まとめ

次: **02_practice/02_langgraph_agent.ipynb** → LangGraph ReAct エージェント